# 3. Patient-Level Data Splitting (Preventing Data Leakage)
## Extract Patient ID from filename (e.g., 'BC003026') and split by patient, NOT by image.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = '/path/to/king-abdulaziz-university-mammogram-dataset' # Update this path

image_paths = []
labels = []
patient_ids = []

class_mapping = {
    'BIRAD1': 0,      # Normal
    'Birad3': 1,      # Benign
    'Birad4': 2,      # Malignant
    'Birad5': 2       # Malignant
}
class_names = ['Normal (BI-RADS 1)', 'Benign (BI-RADS 3)', 'Malignant (BI-RADS 4/5)']

# Regex to extract patient ID (e.g., 'BC003026' from '2013_BC003026_CC_L.jpg')
patient_id_pattern = re.compile(r'(BC\d+)')

for folder_name, label in class_mapping.items():
    folder_path = os.path.join(DATA_DIR, folder_name)
    if not os.path.exists(folder_path):
        continue
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff')):
            filepath = os.path.join(folder_path, filename)
            image_paths.append(filepath)
            labels.append(label)
            
            # Extract patient ID
            match = patient_id_pattern.search(filename)
            if match:
                patient_ids.append(match.group(1))
            else:
                patient_ids.append(filename) # Fallback to filename if pattern fails

print(f"Total images: {len(image_paths)}")
print(f"Unique patients: {len(set(patient_ids))}")

# 1st Split: Train (70%) vs Temp (30%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss1.split(image_paths, labels, groups=patient_ids))

X_train = [image_paths[i] for i in train_idx]
y_train = [labels[i] for i in train_idx]
groups_temp = [patient_ids[i] for i in temp_idx]
labels_temp = [labels[i] for i in temp_idx]
paths_temp = [image_paths[i] for i in temp_idx]

# 2nd Split: Temp into Val (50% of 30% = 15%) and Test (15%)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(paths_temp, labels_temp, groups=groups_temp))

X_val = [paths_temp[i] for i in val_idx]
y_val = [labels_temp[i] for i in val_idx]
X_test = [paths_temp[i] for i in test_idx]
y_test = [labels_temp[i] for i in test_idx]

print(f"\nPatient-Level Split:")
print(f"Train: {len(X_train)} images | Val: {len(X_val)} images | Test: {len(X_test)} images")

# Verify NO patient overlap
train_patients = set([patient_id_pattern.search(os.path.basename(p)).group(1) for p in X_train])
test_patients = set([patient_id_pattern.search(os.path.basename(p)).group(1) for p in X_test])
overlap = train_patients.intersection(test_patients)
print(f"Patient Overlap between Train and Test: {len(overlap)} (Must be 0)")

# 8. Advanced XAI: Grad-CAM vs. Grad-CAM++
## Grad-CAM++ is specifically added to better localize sparse, fine-grained features like microcalcifications.

In [ ]:
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import numpy as np

def get_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-7)
    return heatmap.numpy(), pred_index.numpy()

def get_gradcam_plus_plus(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Grad-CAM++ Implementation (Chattopadhyay et al., 2018)
    Uses higher-order derivatives to better localize multiple disjoint regions (e.g., microcalcifications).
    """
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(model.inputs)
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
        
        # 1st derivative
        grads1 = tape.gradient(class_channel, last_conv_layer_output)
        # 2nd derivative
        grads2 = tape.gradient(grads1, last_conv_layer_output)
        # 3rd derivative
        grads3 = tape.gradient(grads2, last_conv_layer_output)
        
    # Compute alpha weights
    grads1_squared = tf.square(grads1)
    grads1_cubed = grads1_squared * grads1
    
    denom = 2 * grads1_squared + tf.reduce_sum(grads1_cubed * last_conv_layer_output, axis=(1, 2), keepdims=True)
    alpha_num = grads1_squared
    alpha = alpha_num / (denom + 1e-7)
    
    # Global sum pooling of ReLU'd weights * ReLU'd gradients
    valid_alpha = tf.nn.relu(alpha)
    valid_grads = tf.nn.relu(grads1)
    weights = tf.reduce_sum(valid_alpha * valid_grads, axis=(1, 2))
    
    # Final CAM
    cam = tf.reduce_sum(weights[..., tf.newaxis, tf.newaxis] * tf.nn.relu(last_conv_layer_output), axis=-1)
    cam = tf.squeeze(cam)
    cam = tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-7)
    
    return cam.numpy(), pred_index.numpy()

def display_xai_comparison(img_path, model, last_conv_layer_name='top_conv'):
    img = preprocess_mammogram(img_path, (512, 512))
    img_array = np.expand_dims(img, axis=0)
    
    preds = model.predict(img_array, verbose=0)
    pred_class = np.argmax(preds[0])
    confidence = preds[0][pred_class]
    
    # Get both heatmaps
    heatmap_cam, _ = get_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_class)
    heatmap_campp, _ = get_gradcam_plus_plus(img_array, model, last_conv_layer_name, pred_class)
    
    # Colormap application
    jet = plt.colormaps['jet']
    heatmap_cam_colored = jet(heatmap_cam)[..., :3] * 255
    heatmap_campp_colored = jet(heatmap_campp)[..., :3] * 255
    
    # Original image for overlay
    original_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    original_img_resized = cv2.resize(original_img, (512, 512))
    original_img_3ch = np.stack((original_img_resized,) * 3, axis=-1)
    
    # Superimpose
    superimposed_cam = heatmap_cam_colored * 0.4 + original_img_3ch * 0.6
    superimposed_campp = heatmap_campp_colored * 0.4 + original_img_3ch * 0.6
    
    # Plotting
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    
    axes[0].imshow(original_img_resized, cmap='gray')
    axes[0].set_title(f'Original (CLAHE)\nTrue: {class_names[y_test[X_test.index(img_path)]]}', fontsize=10)
    axes[0].axis('off')
    
    axes[1].imshow(heatmap_cam, cmap='jet')
    axes[1].set_title('Standard Grad-CAM', fontsize=10)
    axes[1].axis('off')
    
    axes[2].imshow(heatmap_campp, cmap='jet')
    axes[2].set_title('Grad-CAM++ (Recommended)', fontsize=10)
    axes[2].axis('off')
    
    axes[3].imshow(np.clip(superimposed_campp, 0, 255).astype(np.uint8))
    axes[3].set_title(f'Prediction: {class_names[pred_class]}\nConf: {confidence:.2f}', fontsize=10)
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

# Test on 3 random samples from the strictly separated test set
import random
sample_indices = random.sample(range(len(X_test)), 3)
for i in sample_indices:
    display_xai_comparison(X_test[i], model, last_conv_layer_name='top_conv')